# Fast LmrR EVB/QM workflow

This notebook lives directly in `D:\PhD_Thesis\LmrR_EVB\charges`.

It writes fast-job files to:

`D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm`

It does **not** use the `results` folder.

In [25]:
#!/usr/bin/env python3
r"""
Fast LmrR EVB/QM preparation workflow.

This is meant to replace the slow "run everything in VeloxChem for all steps"
workflow.  By default it only prepares files; it does not launch expensive jobs.

Main ideas
----------
1. Preserve your PDB atom order and atom numbering by converting the existing
   capped PDB files to XYZ and writing a serial-to-XYZ-index map.
2. Prepare xTB jobs for fast pre-optimization of RS/TS/PS structures.
3. Optionally run xTB one step at a time if `xtb` is installed.
4. Export ORCA/Gaussian input files for final QM optimization/frequency jobs.
5. Optionally run the VeloxChem SMILES TS guesser in FAST mode only
   (no relaxed QM scan, no py3Dmol viewer).

Important
---------
- PDB serial numbers are not always the same as XYZ line indices, because the
  capped step-1 PDBs skip some serials. The generated `*_pdb_to_xyz_map.csv`
  files are therefore essential.
- The final barriers still require optimized RS/TS/PS energies and frequencies.
  This script gives fast starting points and final-QM input files.
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import shutil
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


BASE_DIR = Path(r"D:\PhD_Thesis\LmrR_EVB")
CHARGES_DIR = BASE_DIR / "charges"
# Keep this workflow outside the `results` folder by default.
# The user wants code and generated fast-job files directly under `charges`.
RESULTS_DIR = CHARGES_DIR / "fast_lmrr_evb_qm"

HARTREE_TO_KJMOL = 2625.499638


@dataclass(frozen=True)
class PDBAtom:
    serial: int
    name: str
    resname: str
    chain: str
    resid: str
    x: float
    y: float
    z: float
    element: str

    @property
    def res_atom(self) -> str:
        return f"{self.resname}.{self.name}"

    @property
    def pdb_id(self) -> str:
        return f"{self.resid}.{self.name}"


@dataclass(frozen=True)
class BondChange:
    kind: str
    serial_i: int
    serial_j: int
    note: str


@dataclass(frozen=True)
class StructureSet:
    label: str
    rs_pdb: str
    ts_pdb: str
    ps_pdb: str
    charge: int
    multiplicity: int
    bond_changes: tuple[BondChange, ...]
    comment: str


STRUCTURE_SETS: dict[str, StructureSet] = {
    "RS1_1_to_TS1_2": StructureSet(
        label="RS1_1_to_TS1_2",
        rs_pdb="step_1_1_RS.pdb",
        ts_pdb="step_1_1_TS.pdb",
        ps_pdb="step_1_1_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 1, 10, "PAF.N2 attacks ENL.C10; N-C bond formation"),
            BondChange("weakened", 10, 16, "ENL.C10=O1 carbonyl weakens"),
        ),
        comment="Step 1.1 addition. Use TS as a starting TS guess; PS is the tetrahedral state.",
    ),
    "TS1_2_to_PS1_2b": StructureSet(
        label="TS1_2_to_PS1_2b",
        rs_pdb="step_1_2_RS.pdb",
        ts_pdb="step_1_2_TS.pdb",
        ps_pdb="step_1_2_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 1, 29, "PAF N-H breaks"),
            BondChange("formed", 27, 29, "W1 accepts H29"),
            BondChange("broken", 60, 61, "W2 O-H breaks"),
            BondChange("formed", 16, 61, "ENL.O1 accepts H61"),
        ),
        comment="Two-water proton redistribution.",
    ),
    "RS1_2b_to_PS1_3": StructureSet(
        label="RS1_2b_to_PS1_3",
        rs_pdb="step_1_3_RS.pdb",
        ts_pdb="step_1_3_TS.pdb",
        ps_pdb="step_1_3_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 27, 29, "W1 hydronium O-H breaks"),
            BondChange("formed", 16, 29, "carbinolamine O accepts H29"),
            BondChange("broken", 10, 16, "C-O leaving-water bond breaks"),
            BondChange("double-bond formed", 1, 10, "iminium N=C forms"),
        ),
        comment="Dehydration to iminium.",
    ),
    "RS2_1_to_TS2_1a": StructureSet(
        label="RS2_1_to_TS2_1a",
        rs_pdb="step_2_1_RS.pdb",
        ts_pdb="step_2_1_TS.pdb",
        ps_pdb="step_2_1_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 12, 20, "new C-C sigma bond ENL.C12--IND.C3"),
            BondChange("weakened", 1, 10, "iminium N=C weakens"),
            BondChange("strengthened", 10, 11, "C10-C11 bond strengthens"),
        ),
        comment="Friedel-Crafts C-C bond formation.",
    ),
    "TS2_1a_to_PS2_2": StructureSet(
        label="TS2_1a_to_PS2_2",
        # For the combined 2.2 tautomerization, use 2.2a RS as the early state,
        # 2.2b TS as the final-proton-transfer TS guess, and 2.2b PS as PS2.2.
        rs_pdb="step_2_2a_RS.pdb",
        ts_pdb="step_2_2b_TS.pdb",
        ps_pdb="step_2_2b_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 20, 50, "IND.C3-H50 breaks"),
            BondChange("formed", 60, 50, "W2 accepts H50"),
            BondChange("broken", 27, 58, "W1 O-H58 breaks"),
            BondChange("formed", 11, 58, "ENL.C11 accepts H58"),
            BondChange("double-bond formed", 1, 10, "N=C/enamine bond order changes"),
        ),
        comment="Combined 2.2 tautomerization. If you want separate 2.2a/2.2b, split this entry.",
    ),
}


STATE_SMILES = {
    "RS1.1": "CC(=O)NC(Cc1ccc(N)cc1)C(=O)NC.CCC/C=C/C=O",
    "TS1.2": "CC(=O)NC(Cc1ccc([NH2+][CH]([O-])/C=C/CCC)cc1)C(=O)NC",
    "PS1.2b": "CC(=O)NC(Cc1ccc(N[CH](O)/C=C/CCC)cc1)C(=O)NC",
    "RS1.2b": "CC(=O)NC(Cc1ccc(N[CH](O)/C=C/CCC)cc1)C(=O)NC",
    "PS1.3": "CC(=O)NC(Cc1ccc([NH+]=[CH]/C=C/CCC)cc1)C(=O)NC.O",
    "RS2.1": "CC(=O)NC(Cc1ccc([NH+]=[CH]/C=C/CCC)cc1)C(=O)NC.c1ccc2[nH]ccc2c1",
    "TS2.1a": "CC(=O)NC(Cc1ccc(N/C=C/C(CCC)C2C=[NH+]c3ccccc23)cc1)C(=O)NC",
    "PS2.2": "CC(=O)NC(Cc1ccc([NH+]=[CH]CC(CCC)c2c[nH]c3ccccc23)cc1)C(=O)NC",
    "H3O+": "[OH3+]",
    "H2O": "O",
}


SMILES_TRANSITIONS = {
    "RS1_1_to_TS1_2": (["RS1.1"], ["TS1.2"]),
    "TS1_2_to_PS1_2b": (["TS1.2"], ["PS1.2b"]),
    "RS1_2b_to_PS1_3": (["RS1.2b", "H3O+"], ["PS1.3", "H2O"]),
    "RS2_1_to_TS2_1a": (["RS2.1"], ["TS2.1a"]),
    "TS2_1a_to_PS2_2": (["TS2.1a"], ["PS2.2"]),
}


def selected_steps(step_args: Iterable[str] | None) -> list[str]:
    if not step_args:
        return list(STRUCTURE_SETS)
    out: list[str] = []
    for raw in step_args:
        for item in raw.split(","):
            item = item.strip()
            if not item:
                continue
            if item not in STRUCTURE_SETS:
                raise SystemExit(f"Unknown step {item!r}. Available: {', '.join(STRUCTURE_SETS)}")
            out.append(item)
    return out


def infer_element(atom_name: str, explicit: str = "") -> str:
    if explicit:
        return explicit.strip().capitalize()
    letters = "".join(ch for ch in atom_name if ch.isalpha())
    if not letters:
        return "X"
    two = letters[:2].capitalize()
    if two in {"Cl", "Br"}:
        return two
    return letters[0].upper()


def parse_pdb(path: Path) -> list[PDBAtom]:
    atoms: list[PDBAtom] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.startswith(("ATOM  ", "HETATM")):
            continue
        atoms.append(
            PDBAtom(
                serial=int(line[6:11]),
                name=line[12:16].strip(),
                resname=line[17:20].strip(),
                chain=line[21:22].strip(),
                resid=line[22:26].strip(),
                x=float(line[30:38]),
                y=float(line[38:46]),
                z=float(line[46:54]),
                element=infer_element(line[12:16].strip(), line[76:78].strip()),
            )
        )
    if not atoms:
        raise ValueError(f"No atoms found in {path}")
    return atoms


def write_xyz_from_pdb(pdb_path: Path, xyz_path: Path, map_csv_path: Path) -> dict[int, int]:
    atoms = parse_pdb(pdb_path)
    xyz_path.parent.mkdir(parents=True, exist_ok=True)
    serial_to_xyz_index: dict[int, int] = {}
    with xyz_path.open("w", encoding="utf-8") as f:
        f.write(f"{len(atoms)}\n")
        f.write(f"from {pdb_path.name}; PDB serial map in {map_csv_path.name}\n")
        for xyz_index, atom in enumerate(atoms, start=1):
            serial_to_xyz_index[atom.serial] = xyz_index
            f.write(f"{atom.element:<2s} {atom.x:14.8f} {atom.y:14.8f} {atom.z:14.8f}\n")

    with map_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "xyz_index",
                "pdb_serial",
                "pdb_id",
                "res_atom",
                "chain",
                "resid",
                "resname",
                "atom_name",
                "element",
            ],
        )
        writer.writeheader()
        for xyz_index, atom in enumerate(atoms, start=1):
            writer.writerow(
                {
                    "xyz_index": xyz_index,
                    "pdb_serial": atom.serial,
                    "pdb_id": atom.pdb_id,
                    "res_atom": atom.res_atom,
                    "chain": atom.chain,
                    "resid": atom.resid,
                    "resname": atom.resname,
                    "atom_name": atom.name,
                    "element": atom.element,
                }
            )
    return serial_to_xyz_index


def read_xyz(path: Path) -> tuple[list[str], list[tuple[float, float, float]], str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    n = int(lines[0].strip())
    comment = lines[1] if len(lines) > 1 else ""
    labels: list[str] = []
    coords: list[tuple[float, float, float]] = []
    for line in lines[2 : 2 + n]:
        parts = line.split()
        labels.append(parts[0])
        coords.append((float(parts[1]), float(parts[2]), float(parts[3])))
    return labels, coords, comment


def distance_from_xyz(xyz_path: Path, idx_i: int, idx_j: int) -> float:
    _, coords, _ = read_xyz(xyz_path)
    xi, yi, zi = coords[idx_i - 1]
    xj, yj, zj = coords[idx_j - 1]
    return math.sqrt((xi - xj) ** 2 + (yi - yj) ** 2 + (zi - zj) ** 2)


def write_xtb_constraint(path: Path, constraints: list[tuple[int, int, float]]) -> None:
    lines = ["$constrain"]
    for i, j, dist in constraints:
        lines.append(f"  distance: {i}, {j}, {dist:.4f}")
    lines.append("$end")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_xtb_runner(
    run_path: Path,
    xyz_path: Path,
    charge: int,
    multiplicity: int,
    constraint_file: Path | None,
    output_prefix: str,
) -> None:
    uhf = max(0, multiplicity - 1)
    cmd = [
        "xtb",
        xyz_path.name,
        "--gfn",
        "2",
        "--opt",
        "tight",
        "--chrg",
        str(charge),
        "--uhf",
        str(uhf),
    ]
    if constraint_file is not None:
        cmd.extend(["--input", constraint_file.name])
    command = " ".join(cmd)
    run_path.write_text(
        "\n".join(
            [
                "$ErrorActionPreference = 'Stop'",
                f"Set-Location -LiteralPath '{run_path.parent}'",
                command + f" *> {output_prefix}.xtb.log",
                "if (Test-Path xtbopt.xyz) {",
                f"    Copy-Item -LiteralPath xtbopt.xyz -Destination {output_prefix}_xtbopt.xyz -Force",
                "}",
                "if (Test-Path xtbopt.log) {",
                f"    Copy-Item -LiteralPath xtbopt.log -Destination {output_prefix}_xtbopt.log -Force",
                "}",
                "if (Test-Path xtbrestart) {",
                f"    Copy-Item -LiteralPath xtbrestart -Destination {output_prefix}.xtbrestart -Force",
                "}",
                "",
            ]
        ),
        encoding="utf-8",
    )


def write_orca_input(
    inp_path: Path,
    xyz_path: Path,
    charge: int,
    multiplicity: int,
    job_type: str,
    nprocs: int = 8,
    memory_mb: int = 3000,
) -> None:
    labels, coords, _ = read_xyz(xyz_path)
    if job_type == "ts":
        route = "! B3LYP D3BJ def2-SVP OptTS NumFreq TightSCF"
    elif job_type == "freq":
        route = "! B3LYP D3BJ def2-SVP Freq TightSCF"
    else:
        route = "! B3LYP D3BJ def2-SVP Opt Freq TightSCF"
    lines = [
        route,
        f"%pal nprocs {nprocs} end",
        f"%maxcore {memory_mb}",
        "",
        f"* xyz {charge} {multiplicity}",
    ]
    for label, (x, y, z) in zip(labels, coords):
        lines.append(f"  {label:<2s} {x:14.8f} {y:14.8f} {z:14.8f}")
    lines.append("*")
    lines.append("")
    inp_path.write_text("\n".join(lines), encoding="utf-8")


def write_gaussian_input(
    gjf_path: Path,
    xyz_path: Path,
    charge: int,
    multiplicity: int,
    job_type: str,
    nprocs: int = 8,
    memory_gb: int = 16,
) -> None:
    labels, coords, _ = read_xyz(xyz_path)
    opt_keyword = "Opt=(TS,CalcFC,NoEigenTest) Freq" if job_type == "ts" else "Opt Freq"
    lines = [
        f"%nprocshared={nprocs}",
        f"%mem={memory_gb}GB",
        f"#p B3LYP/def2SVP EmpiricalDispersion=GD3BJ {opt_keyword}",
        "",
        f"{xyz_path.stem} {job_type}",
        "",
        f"{charge} {multiplicity}",
    ]
    for label, (x, y, z) in zip(labels, coords):
        lines.append(f"{label:<2s} {x:14.8f} {y:14.8f} {z:14.8f}")
    lines.extend(["", ""])
    gjf_path.write_text("\n".join(lines), encoding="utf-8")


def prepare_pdb_preserving_jobs(steps: list[str], overwrite: bool = True) -> None:
    prep_dir = RESULTS_DIR / "pdb_preserving_xtb_qm_inputs"
    prep_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []

    for step in steps:
        cfg = STRUCTURE_SETS[step]
        step_dir = prep_dir / step
        step_dir.mkdir(parents=True, exist_ok=True)
        serial_maps: dict[str, dict[int, int]] = {}

        for role, pdb_name, qm_job_type in [
            ("RS", cfg.rs_pdb, "min"),
            ("TS", cfg.ts_pdb, "ts"),
            ("PS", cfg.ps_pdb, "min"),
        ]:
            pdb_path = CHARGES_DIR / pdb_name
            if not pdb_path.exists():
                raise FileNotFoundError(pdb_path)
            xyz_path = step_dir / f"{step}_{role}.xyz"
            map_csv = step_dir / f"{step}_{role}_pdb_to_xyz_map.csv"
            if overwrite or not xyz_path.exists():
                serial_maps[role] = write_xyz_from_pdb(pdb_path, xyz_path, map_csv)
            else:
                # Read existing map.
                serial_maps[role] = {}
                with map_csv.open(newline="", encoding="utf-8") as f:
                    for row in csv.DictReader(f):
                        serial_maps[role][int(row["pdb_serial"])] = int(row["xyz_index"])

            write_orca_input(step_dir / f"{step}_{role}.inp", xyz_path, cfg.charge, cfg.multiplicity, qm_job_type)
            write_gaussian_input(step_dir / f"{step}_{role}.gjf", xyz_path, cfg.charge, cfg.multiplicity, qm_job_type)
            write_xtb_runner(
                step_dir / f"run_xtb_{role}.ps1",
                xyz_path,
                cfg.charge,
                cfg.multiplicity,
                constraint_file=None,
                output_prefix=f"{step}_{role}",
            )

            summary_rows.append(
                {
                    "step": step,
                    "role": role,
                    "pdb": str(pdb_path),
                    "xyz": str(xyz_path),
                    "map_csv": str(map_csv),
                    "orca_input": str(step_dir / f"{step}_{role}.inp"),
                    "gaussian_input": str(step_dir / f"{step}_{role}.gjf"),
                    "xtb_runner": str(step_dir / f"run_xtb_{role}.ps1"),
                    "charge": cfg.charge,
                    "multiplicity": cfg.multiplicity,
                    "comment": cfg.comment,
                }
            )

        # Write constrained xTB TS preoptimization for the TS structure.
        ts_xyz = step_dir / f"{step}_TS.xyz"
        ts_map = serial_maps["TS"]
        constraints: list[tuple[int, int, float]] = []
        for change in cfg.bond_changes:
            if change.kind in {"formed", "broken", "double-bond formed"}:
                if change.serial_i in ts_map and change.serial_j in ts_map:
                    i_xyz = ts_map[change.serial_i]
                    j_xyz = ts_map[change.serial_j]
                    dist = distance_from_xyz(ts_xyz, i_xyz, j_xyz)
                    constraints.append((i_xyz, j_xyz, dist))
        if constraints:
            constraint_file = step_dir / f"{step}_TS_xtb_constraints.inp"
            write_xtb_constraint(constraint_file, constraints)
            write_xtb_runner(
                step_dir / "run_xtb_TS_constrained.ps1",
                ts_xyz,
                cfg.charge,
                cfg.multiplicity,
                constraint_file=constraint_file,
                output_prefix=f"{step}_TS_constrained",
            )

        # Human-readable bond map for this step.
        atoms = parse_pdb(CHARGES_DIR / cfg.ts_pdb)
        by_serial = {a.serial: a for a in atoms}
        bond_lines = [
            f"# {step}",
            f"# {cfg.comment}",
            "# PDB serial -> atom identity; XYZ index is taken from TS map",
        ]
        for change in cfg.bond_changes:
            ai = by_serial[change.serial_i]
            aj = by_serial[change.serial_j]
            bond_lines.append(
                f"{change.kind:18s} PDB {change.serial_i:>3}-{change.serial_j:<3} "
                f"XYZ {ts_map[change.serial_i]:>3}-{ts_map[change.serial_j]:<3} "
                f"{ai.res_atom} -- {aj.res_atom} ; {change.note}"
            )
        (step_dir / f"{step}_bond_map.txt").write_text("\n".join(bond_lines) + "\n", encoding="utf-8")

    summary_csv = prep_dir / "prepared_fast_jobs_summary.csv"
    with summary_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "step",
                "role",
                "pdb",
                "xyz",
                "map_csv",
                "orca_input",
                "gaussian_input",
                "xtb_runner",
                "charge",
                "multiplicity",
                "comment",
            ],
        )
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"Prepared PDB-preserving xTB/QM inputs in: {prep_dir}")
    print(f"Summary: {summary_csv}")


def run_xtb_jobs(steps: list[str], roles: list[str]) -> None:
    xtb = shutil.which("xtb")
    if not xtb:
        raise SystemExit(
            "Could not find xtb on PATH. Install xTB or use the prepared run_xtb_*.ps1 files on a machine with xTB."
        )
    prep_dir = RESULTS_DIR / "pdb_preserving_xtb_qm_inputs"
    for step in steps:
        step_dir = prep_dir / step
        for role in roles:
            runner = step_dir / f"run_xtb_{role}.ps1"
            if role == "TS_constrained":
                runner = step_dir / "run_xtb_TS_constrained.ps1"
            if not runner.exists():
                print(f"Skipping missing runner: {runner}")
                continue
            print(f"Running {runner}")
            subprocess.run(["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File", str(runner)], check=True)


def _safe_float(value):
    try:
        return float(value)
    except Exception:
        return None


def _scan_items(result):
    scan = result.get("scan", {}) if isinstance(result, dict) else {}
    if not isinstance(scan, dict):
        return []
    items = []
    for lam, records in scan.items():
        if isinstance(records, dict):
            records = [records]
        for idx, rec in enumerate(records or []):
            if isinstance(rec, dict):
                items.append((lam, idx, rec))
    return items


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def run_veloxchem_fast(steps: list[str]) -> None:
    try:
        import veloxchem as vlx
    except Exception as exc:
        raise SystemExit(f"VeloxChem is not importable in this Python environment: {exc}") from exc

    out_dir = RESULTS_DIR / "veloxchem_fast_smiles_guesses"
    out_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []

    for step in steps:
        reactant_labels, product_labels = SMILES_TRANSITIONS[step]
        reactants = [vlx.Molecule.read_smiles(STATE_SMILES[label]) for label in reactant_labels]
        products = [vlx.Molecule.read_smiles(STATE_SMILES[label]) for label in product_labels]

        guesser = vlx.TransitionStateGuesser()
        # The key speed choices: no QM rescoring/relaxed QM scan and no viewer.
        for attr, value in [
            ("scf_scan", False),
            ("do_qm_scan", False),
            ("qm_scan", False),
            ("optimize_ts", False),
            ("do_ts_optimization", False),
            ("mute_ff_build", True),
        ]:
            if hasattr(guesser, attr):
                setattr(guesser, attr, value)

        print(f"\n=== VeloxChem FAST guess: {step} ===")
        result = guesser.find_transition_state(reactants, products)
        step_dir = out_dir / step
        step_dir.mkdir(parents=True, exist_ok=True)

        for key in ["max_mm_xyz", "max_qm_xyz"]:
            if isinstance(result, dict) and key in result:
                write_text(step_dir / f"{step}_{key}.xyz", result[key])

        scan_rows = []
        for lam, idx, rec in _scan_items(result):
            xyz = rec.get("qm_xyz") or rec.get("mm_xyz") or rec.get("xyz")
            if xyz:
                write_text(step_dir / f"lambda_{float(lam):.2f}_conf_{idx}.xyz", xyz)
            scan_rows.append(
                {
                    "step": step,
                    "lambda": float(lam),
                    "conformer_index": idx,
                    "mm_energy": rec.get("mm_energy", ""),
                    "qm_energy": rec.get("qm_energy", ""),
                    "has_xyz": bool(xyz),
                }
            )
        if scan_rows:
            write_csv(
                step_dir / f"{step}_veloxchem_fast_scan.csv",
                scan_rows,
                ["step", "lambda", "conformer_index", "mm_energy", "qm_energy", "has_xyz"],
            )

        summary_rows.append(
            {
                "step": step,
                "reactants": " + ".join(reactant_labels),
                "products": " + ".join(product_labels),
                "result_keys": ", ".join(result.keys()) if isinstance(result, dict) else "",
                "max_mm_lambda": result.get("max_mm_lambda", "") if isinstance(result, dict) else "",
                "max_qm_lambda": result.get("max_qm_lambda", "") if isinstance(result, dict) else "",
                "scan_points": len(scan_rows),
                "folder": str(step_dir),
            }
        )

    write_csv(
        out_dir / "veloxchem_fast_summary.csv",
        summary_rows,
        ["step", "reactants", "products", "result_keys", "max_mm_lambda", "max_qm_lambda", "scan_points", "folder"],
    )
    print(f"\nVeloxChem fast outputs: {out_dir}")


def write_readme() -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    readme = RESULTS_DIR / "README_fast_workflow.md"
    readme.write_text(
        """# Fast LmrR EVB/QM workflow

This folder is generated by `fast_lmrr_evb_qm_workflow.py`.

Recommended fast route:

1. Prepare PDB-preserving XYZ, xTB runners, and final-QM input files:

   ```powershell
   python fast_lmrr_evb_qm_workflow.py --prepare
   ```

2. Run xTB only for one step/role at a time, for example:

   ```powershell
   python fast_lmrr_evb_qm_workflow.py --run-xtb --steps RS2_1_to_TS2_1a --roles RS,TS,PS
   ```

   Or run the generated `run_xtb_*.ps1` files manually inside each step folder.

3. Use the generated ORCA `.inp` or Gaussian `.gjf` files for final B3LYP-D/def2-SVP
   optimization/frequency calculations.

4. Use VeloxChem only for quick SMILES TS guesses:

   ```powershell
   python fast_lmrr_evb_qm_workflow.py --veloxchem-fast --steps RS2_1_to_TS2_1a
   ```

Do not run all relaxed QM scans for all five steps in one notebook. That is the slow
path that can consume days without useful intermediate stopping points.
""",
        encoding="utf-8",
    )


def main() -> None:
    parser = argparse.ArgumentParser(description="Fast LmrR EVB/QM preparation workflow.")
    parser.add_argument("--steps", nargs="*", help="Step labels, comma-separated or space-separated.")
    parser.add_argument("--prepare", action="store_true", help="Prepare PDB-preserving XYZ/xTB/QM input files.")
    parser.add_argument("--run-xtb", action="store_true", help="Run prepared xTB jobs. Requires xtb on PATH.")
    parser.add_argument(
        "--roles",
        default="RS,TS,PS",
        help="Roles for --run-xtb: RS,TS,PS,TS_constrained. Default: RS,TS,PS.",
    )
    parser.add_argument("--veloxchem-fast", action="store_true", help="Run fast VeloxChem SMILES TS guesses only.")
    parser.add_argument("--list-steps", action="store_true", help="Print available step labels and exit.")
    args = parser.parse_args()

    if args.list_steps:
        print("\n".join(STRUCTURE_SETS))
        return

    steps = selected_steps(args.steps)
    write_readme()

    if not args.prepare and not args.run_xtb and not args.veloxchem_fast:
        print("No action requested. Preparing files by default.")
        args.prepare = True

    if args.prepare:
        prepare_pdb_preserving_jobs(steps)
    if args.run_xtb:
        roles = [x.strip() for x in args.roles.split(",") if x.strip()]
        run_xtb_jobs(steps, roles)
    if args.veloxchem_fast:
        run_veloxchem_fast(steps)

## Check xTB path

Run this first. If it prints `None`, xTB is not installed or not on PATH.

In [26]:
import os, shutil
from pathlib import Path

# If xTB is installed but not on PATH, paste the full xtb.exe path here:
XTB_EXE = r""  # example: r"C:\Users\Nayanika\miniconda3\envs\xtb\Library\bin\xtb.exe"

if XTB_EXE and Path(XTB_EXE).exists():
    os.environ["PATH"] = str(Path(XTB_EXE).parent) + os.pathsep + os.environ["PATH"]

print("xtb found:", shutil.which("xtb"))
print("workflow output folder:", RESULTS_DIR)

xtb found: C:\Users\Nayanika\miniconda3\envs\quimica-echem\Library\bin\xtb.EXE
workflow output folder: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm


## Available EVB steps and PDB files

In [27]:
for step, cfg in STRUCTURE_SETS.items():
    print(step)
    print("  RS:", cfg.rs_pdb)
    print("  TS:", cfg.ts_pdb)
    print("  PS:", cfg.ps_pdb)
    print()

RS1_1_to_TS1_2
  RS: step_1_1_RS.pdb
  TS: step_1_1_TS.pdb
  PS: step_1_1_PS.pdb

TS1_2_to_PS1_2b
  RS: step_1_2_RS.pdb
  TS: step_1_2_TS.pdb
  PS: step_1_2_PS.pdb

RS1_2b_to_PS1_3
  RS: step_1_3_RS.pdb
  TS: step_1_3_TS.pdb
  PS: step_1_3_PS.pdb

RS2_1_to_TS2_1a
  RS: step_2_1_RS.pdb
  TS: step_2_1_TS.pdb
  PS: step_2_1_PS.pdb

TS2_1a_to_PS2_2
  RS: step_2_2a_RS.pdb
  TS: step_2_2b_TS.pdb
  PS: step_2_2b_PS.pdb



## Prepare files only

Creates XYZ/maps/xTB runners/ORCA/Gaussian inputs under `charges
ast_lmrr_evb_qm`. Does not run xTB/QM.

In [28]:
import os, shutil
from pathlib import Path

XTB_EXE = r"C:\Users\Nayanika\miniconda3\envs\quimica-echem\Library\bin\xtb.exe"

os.environ["PATH"] = str(Path(XTB_EXE).parent) + os.pathsep + os.environ["PATH"]

print("xtb found:", shutil.which("xtb"))

prepare_pdb_preserving_jobs(list(STRUCTURE_SETS))

xtb found: C:\Users\Nayanika\miniconda3\envs\quimica-echem\Library\bin\xtb.EXE
Prepared PDB-preserving xTB/QM inputs in: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm\pdb_preserving_xtb_qm_inputs
Summary: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm\pdb_preserving_xtb_qm_inputs\prepared_fast_jobs_summary.csv


In [29]:
import shutil
import os
from pathlib import Path

# Where you extracted the official xTB
XTB_DIR = Path(r"C:\Users\Nayanika\Downloads\xtb-6.7.1pre-windows-x86_64")

# Search for xtb.exe recursively
xtb_exe = None
for exe in XTB_DIR.rglob("xtb.exe"):
    xtb_exe = exe
    break

if xtb_exe is None:
    # Also check if xtb is in PATH
    path_xtb = shutil.which("xtb")
    if path_xtb:
        print(f"Using system xtb at: {path_xtb} (may be the Conda version).")
        XTB_EXE = Path(path_xtb)
    else:
        raise FileNotFoundError(f"xtb.exe not found in {XTB_DIR}. Please check the folder.")
else:
    XTB_EXE = xtb_exe
    print(f"Found official xTB at: {XTB_EXE}")

# Verify it works
if XTB_EXE.exists():
    print("✅ xTB binary found.")
else:
    raise FileNotFoundError("No xtb.exe found.")

Found official xTB at: C:\Users\Nayanika\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.exe
✅ xTB binary found.


In [30]:
# ================================================================
# RUN xTB + COLLECT QM REFERENCE DATA FOR EVB
# ================================================================
#
# This is the main post-preparation workflow.
#
# For every step / RS / TS / PS it creates:
#
#   step_x/
#       RS/
#           geometry/
#           energy/
#           charges/
#           frequencies/
#           input/
#       TS/
#       PS/
#
# and global:
#
#   summaries/
#       qm_state_summary.csv
#       qm_barriers_for_evb_comparison.csv
#       qm_charge_changes.csv
#       qm_mapping_validation.csv
#
# IMPORTANT:
# - xTB is a preparation/reference stage.
# - DFT energies should be used as the higher-level QM reference for EVB.
# - The QM barrier is ΔE‡ = E_TS - E_RS, not an EVB ΔG‡.
# - Existing charge files are copied/organized; the script does not invent charges.

import os
import re
import csv
import shutil
import subprocess
from pathlib import Path

HARTREE_TO_KJMOL = 2625.499638

EVB_QM_ROOT = RESULTS_DIR / "qm_reference_for_evb"
SUMMARY_DIR = EVB_QM_ROOT / "summaries"
INPUT_DIR = EVB_QM_ROOT / "inputs"

for d in [EVB_QM_ROOT, SUMMARY_DIR, INPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def _copy_if_exists(src: Path, dst: Path) -> bool:
    if src.exists() and src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True
    return False


def parse_xtb_energy(logfile: Path):
    """Return the final xTB total energy in Hartree if present."""
    if not logfile.exists():
        return None
    text = logfile.read_text(encoding="utf-8", errors="replace")

    # Typical xTB line:
    # TOTAL ENERGY       -123.456789 Eh
    matches = re.findall(
        r"TOTAL ENERGY\s+([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)\s+Eh",
        text,
        flags=re.I,
    )
    if matches:
        return float(matches[-1])

    # Fallback for slightly different formatting.
    matches = re.findall(
        r"TOTAL ENERGY\s*[:=]?\s*([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        text,
        flags=re.I,
    )
    return float(matches[-1]) if matches else None


def parse_orca_energy(out: Path):
    if not out.exists():
        return None
    text = out.read_text(encoding="utf-8", errors="replace")
    vals = re.findall(
        r"FINAL SINGLE POINT ENERGY\s+([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        text,
        flags=re.I,
    )
    return float(vals[-1]) if vals else None


def parse_gaussian_energy(out: Path):
    if not out.exists():
        return None
    text = out.read_text(encoding="utf-8", errors="replace")
    vals = re.findall(
        r"SCF Done:\s+E\([^)]+\)\s+=\s+([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        text,
        flags=re.I,
    )
    return float(vals[-1]) if vals else None


def parse_mulliken_from_orca(out: Path):
    """Extract ORCA Mulliken charges into a simple CSV."""
    if not out.exists():
        return None

    lines = out.read_text(encoding="utf-8", errors="replace").splitlines()
    start = None

    for i, line in enumerate(lines):
        if "MULLIKEN ATOMIC CHARGES" in line.upper():
            start = i + 1

    if start is None:
        return None

    rows = []
    for line in lines[start:]:
        s = line.strip()
        if not s:
            if rows:
                break
            continue
        if set(s) <= {"-"}:
            continue
        m = re.match(r"^\s*(\d+)\s+([A-Za-z]+)\s+([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)", line)
        if m:
            rows.append((int(m.group(1)), m.group(2), float(m.group(3))))
        elif rows and ("SUM OF" in s.upper() or "MULLIKEN" in s.upper()):
            break

    return rows or None


def parse_gaussian_mulliken(out: Path):
    """Extract the last Mulliken charge table from a Gaussian output."""
    if not out.exists():
        return None

    lines = out.read_text(encoding="utf-8", errors="replace").splitlines()
    starts = [i for i, line in enumerate(lines) if "Mulliken charges:" in line]
    if not starts:
        return None

    start = starts[-1] + 1
    rows = []
    for line in lines[start:]:
        m = re.match(
            r"^\s*(\d+)\s+([A-Za-z]+)\s+([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
            line,
        )
        if m:
            rows.append((int(m.group(1)), m.group(2), float(m.group(3))))
        elif rows and line.strip() == "":
            break

    return rows or None


def write_charge_csv(rows, path: Path):
    if not rows:
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["qm_index", "element", "charge_e"])
        w.writerows(rows)
    return True


def find_existing_charge_file(step: str, role: str, step_dir: Path):
    """
    Look for charge files already generated by the user's QM workflow.

    Priority:
      1. step/role-specific files in the prepared directory
      2. files directly in charges/
    """
    candidates = []

    patterns = [
        f"*{step}*{role}*charges*",
        f"*{step}*{role}*.chg",
        f"*{step}*{role}*.dat",
        f"*{step}*{role}*.txt",
        f"*{step}_{role}*charges*",
        f"*{step}_{role}*.chg",
    ]

    for pattern in patterns:
        candidates.extend(step_dir.glob(pattern))
        candidates.extend(CHARGES_DIR.glob(pattern))

    # Avoid copying our own output folders back into themselves.
    candidates = [
        x for x in candidates
        if x.is_file() and EVB_QM_ROOT not in x.parents
    ]

    return candidates[0] if candidates else None


def make_state_folder(step: str, role: str):
    state = EVB_QM_ROOT / step / role
    dirs = {
        "root": state,
        "geometry": state / "geometry",
        "energy": state / "energy",
        "charges": state / "charges",
        "frequencies": state / "frequencies",
        "input": state / "input",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def collect_one_state(step: str, role: str):
    """Collect all available QM information for one RS/TS/PS state."""
    cfg = STRUCTURE_SETS[step]
    dirs = make_state_folder(step, role)

    prep = RESULTS_DIR / "pdb_preserving_xtb_qm_inputs" / step

    xyz = prep / f"{step}_{role}.xyz"
    xtbopt = prep / f"{step}_{role}_xtbopt.xyz"
    xtb_log = prep / f"{step}_{role}_xtbopt.log"
    xtb_out = prep / f"{step}_{role}_xtb.out"

    # Geometry
    geometry_source = None
    if xtbopt.exists():
        geometry_source = xtbopt
    elif xyz.exists():
        geometry_source = xyz

    if geometry_source:
        _copy_if_exists(
            geometry_source,
            dirs["geometry"] / f"{step}_{role}_geometry.xyz"
        )

    # Inputs / mapping
    for pattern in [
        f"{step}_{role}.inp",
        f"{step}_{role}.gjf",
        f"{step}_{role}_pdb_to_xyz_map.csv",
    ]:
        _copy_if_exists(prep / pattern, dirs["input"] / pattern)

    # xTB log/output
    for src in [xtb_log, xtb_out]:
        if src.exists():
            _copy_if_exists(src, dirs["energy"] / src.name)

    # xTB energy
    energy_hartree = None
    for logfile in [xtb_log, xtb_out]:
        val = parse_xtb_energy(logfile)
        if val is not None:
            energy_hartree = val
            break

    energy_source = "xTB"
    energy_file = dirs["energy"] / f"{step}_{role}_energies.csv"

    # Look for ORCA/Gaussian output if present.
    orca_outputs = list(prep.glob(f"*{role}*.out")) + list(prep.glob(f"*{role}*.log"))
    gaussian_outputs = list(prep.glob(f"*{role}*.log")) + list(prep.glob(f"*{role}*.out"))

    orca_energy = None
    gaussian_energy = None
    orca_file = None
    gaussian_file = None

    for f in orca_outputs:
        v = parse_orca_energy(f)
        if v is not None:
            orca_energy, orca_file = v, f
            break

    for f in gaussian_outputs:
        v = parse_gaussian_energy(f)
        if v is not None:
            gaussian_energy, gaussian_file = v, f
            break

    # Prefer higher-level QM energy if it is actually available.
    final_energy = energy_hartree
    final_method = "xTB"
    final_source = xtb_log if energy_hartree is not None else None

    if orca_energy is not None:
        final_energy = orca_energy
        final_method = "ORCA"
        final_source = orca_file
    elif gaussian_energy is not None:
        final_energy = gaussian_energy
        final_method = "Gaussian"
        final_source = gaussian_file

    if final_source:
        _copy_if_exists(
            final_source,
            dirs["energy"] / Path(final_source).name
        )

    with energy_file.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["step", "role", "method", "energy_hartree", "energy_kjmol", "source"])
        if final_energy is not None:
            w.writerow([
                step,
                role,
                final_method,
                f"{final_energy:.12f}",
                f"{final_energy * HARTREE_TO_KJMOL:.6f}",
                str(final_source) if final_source else "",
            ])

    # Charges: first parse DFT output if available, otherwise organize existing file.
    charge_csv = dirs["charges"] / f"{step}_{role}_mulliken.csv"
    charge_source = ""

    if orca_file:
        rows = parse_mulliken_from_orca(orca_file)
        if rows and write_charge_csv(rows, charge_csv):
            charge_source = f"ORCA:{orca_file.name}"

    if not charge_source and gaussian_file:
        rows = parse_gaussian_mulliken(gaussian_file)
        if rows and write_charge_csv(rows, charge_csv):
            charge_source = f"Gaussian:{gaussian_file.name}"

    if not charge_source:
        existing = find_existing_charge_file(step, role, prep)
        if existing:
            dst = dirs["charges"] / existing.name
            _copy_if_exists(existing, dst)
            charge_source = f"existing:{existing}"

    # xTB's charges file is commonly produced when population analysis is requested.
    if not charge_source:
        for candidate in [
            prep / "charges",
            prep / f"{step}_{role}_charges",
            prep / f"{step}_{role}_charges.dat",
        ]:
            if candidate.exists() and candidate.is_file():
                _copy_if_exists(candidate, dirs["charges"] / candidate.name)
                charge_source = f"xTB:{candidate.name}"
                break

    # Frequencies: copy known frequency files if the user has them.
    frequency_sources = []
    for pattern in [
        f"*{step}*{role}*freq*",
        f"*{step}*{role}*.hess",
        f"*{step}*{role}*.out",
        f"*{step}*{role}*.log",
    ]:
        for f in prep.glob(pattern):
            if f.is_file():
                # Do not duplicate energy outputs unnecessarily.
                if "freq" in f.name.lower() or f.suffix.lower() in {".hess"}:
                    dst = dirs["frequencies"] / f.name
                    if _copy_if_exists(f, dst):
                        frequency_sources.append(f.name)

    return {
        "step": step,
        "role": role,
        "charge": cfg.charge,
        "multiplicity": cfg.multiplicity,
        "energy_method": final_method if final_energy is not None else "",
        "energy_hartree": final_energy,
        "energy_kjmol": final_energy * HARTREE_TO_KJMOL if final_energy is not None else None,
        "geometry": str(dirs["geometry"]),
        "energy_folder": str(dirs["energy"]),
        "charges_folder": str(dirs["charges"]),
        "frequency_folder": str(dirs["frequencies"]),
        "input_folder": str(dirs["input"]),
        "charge_source": charge_source,
        "frequency_files": ";".join(frequency_sources),
    }


def collect_qm_reference(steps=None):
    """Build the complete EVB-ready QM reference tree."""
    steps = steps or list(STRUCTURE_SETS)

    rows = []
    for step in steps:
        for role in ["RS", "TS", "PS"]:
            row = collect_one_state(step, role)
            rows.append(row)

    fields = [
        "step", "role", "charge", "multiplicity",
        "energy_method", "energy_hartree", "energy_kjmol",
        "geometry", "energy_folder", "charges_folder",
        "frequency_folder", "input_folder",
        "charge_source", "frequency_files",
    ]

    with (SUMMARY_DIR / "qm_state_summary.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

    # Barrier / reaction-energy table.
    by_step = {}
    for r in rows:
        by_step.setdefault(r["step"], {})[r["role"]] = r

    barrier_rows = []
    for step in steps:
        d = by_step.get(step, {})
        rs = d.get("RS", {})
        ts = d.get("TS", {})
        ps = d.get("PS", {})

        er = rs.get("energy_hartree")
        et = ts.get("energy_hartree")
        ep = ps.get("energy_hartree")

        barrier = (et - er) * HARTREE_TO_KJMOL if er is not None and et is not None else None
        reaction = (ep - er) * HARTREE_TO_KJMOL if er is not None and ep is not None else None

        barrier_rows.append({
            "step": step,
            "energy_method_RS": rs.get("energy_method", ""),
            "energy_method_TS": ts.get("energy_method", ""),
            "energy_method_PS": ps.get("energy_method", ""),
            "RS_energy_Ha": er,
            "TS_energy_Ha": et,
            "PS_energy_Ha": ep,
            "QM_DeltaE_activation_kJmol": barrier,
            "QM_DeltaE_reaction_kJmol": reaction,
            "EVB_DeltaG_activation_kJmol": "",
            "EVB_DeltaG_reaction_kJmol": "",
            "EVB_minus_QM_barrier_kJmol": "",
            "notes": "QM ΔE‡ is an electronic/reference-model quantity; EVB ΔG‡ is the later protein/solvent free-energy quantity.",
        })

    barrier_fields = list(barrier_rows[0].keys())
    with (SUMMARY_DIR / "qm_barriers_for_evb_comparison.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=barrier_fields)
        w.writeheader()
        w.writerows(barrier_rows)

    print("\n============================================================")
    print("QM reference dataset prepared for EVB")
    print("============================================================")
    print(f"Root: {EVB_QM_ROOT}")
    print(f"State summary: {SUMMARY_DIR / 'qm_state_summary.csv'}")
    print(f"Barrier table: {SUMMARY_DIR / 'qm_barriers_for_evb_comparison.csv'}")

    for r in rows:
        status = "ENERGY" if r["energy_hartree"] is not None else "NO ENERGY"
        charge_status = "CHARGES" if r["charge_source"] else "NO CHARGES"
        print(f"{r['step']:28s} {r['role']:3s}  {status:10s}  {charge_status}")


def write_evb_comparison_template():
    """Create a blank table that can later be filled with EVB results."""
    path = SUMMARY_DIR / "evb_comparison_template.csv"
    fields = [
        "step",
        "QM_DeltaE_activation_kJmol",
        "QM_DeltaE_reaction_kJmol",
        "EVB_DeltaG_activation_kJmol",
        "EVB_DeltaG_reaction_kJmol",
        "EVB_minus_QM_barrier_kJmol",
        "interpretation",
    ]
    if not path.exists():
        with path.open("w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(fields)
    print(f"EVB comparison template: {path}")


# Run after your xTB/DFT jobs have produced outputs.
# collect_qm_reference(list(STRUCTURE_SETS))
# write_evb_comparison_template()


In [31]:
# The old duplicate xTB optimization cell has been replaced by the consolidated collector in cell 9.


In [32]:
step_dir = RESULTS_DIR / "pdb_preserving_xtb_qm_inputs" / "RS1_1_to_TS1_2"
log_file = step_dir / "RS1_1_to_TS1_2_RS_xtbopt.log"
if log_file.exists():
    content = log_file.read_text(encoding="utf-8", errors="replace")
    lines = content.splitlines()
    print("=== Last 30 lines of log ===")
    for line in lines[-30:]:
        print(line)
else:
    print("Log file not found.")

=== Last 30 lines of log ===
H            7.94180333499612        0.00833819319186       -0.85440517847935
H            1.26281347179888        0.39786349040457       -3.59977953183357
H            3.78751891595479        2.19234889152618       -3.67783574117594
H            0.89763252979098        2.56749084513284       -2.74626538119663
H            2.23505203646819        4.33090407007171       -1.49239254328328
H            3.33872420728040        4.43606399536270       -2.86916281271374
H            1.42741552540106        5.17773455202592       -4.30504744514146
H            0.34067276854500        5.05859756162318       -2.91947059541216
H            1.08496531319251        7.40604448642241       -3.27330494846540
H            1.67082667403588        6.79914202248459       -1.72384435733588
H            2.76823941707454        6.90937112598954       -3.09953503194539
H            0.16923020507896       -2.62516843365308       -3.09229798864648
H           -1.33917348203620      

In [33]:
# ================================================================
# FINAL QM BARRIER TABLE FOR LATER EVB COMPARISON
# ================================================================
#
# Run this after:
#   collect_qm_reference(list(STRUCTURE_SETS))
#
# The resulting CSV contains blank EVB columns intentionally.
# Later we can fill those with the EVB ΔG‡ values.

import pandas as pd

barrier_csv = SUMMARY_DIR / "qm_barriers_for_evb_comparison.csv"

if barrier_csv.exists():
    df_qm = pd.read_csv(barrier_csv)

    display(
        df_qm[
            [
                "step",
                "energy_method_RS",
                "energy_method_TS",
                "energy_method_PS",
                "QM_DeltaE_activation_kJmol",
                "QM_DeltaE_reaction_kJmol",
                "EVB_DeltaG_activation_kJmol",
                "EVB_minus_QM_barrier_kJmol",
            ]
        ]
    )
else:
    print("Run collect_qm_reference(list(STRUCTURE_SETS)) first.")


,step,energy_method_RS,energy_method_TS,energy_method_PS,QM_DeltaE_activation_kJmol,QM_DeltaE_reaction_kJmol,EVB_DeltaG_activation_kJmol,EVB_minus_QM_barrier_kJmol
0,RS1_1_to_TS1_2,xTB,xTB,xTB,0.198298,9.243404,NaN,NaN
1,TS1_2_to_PS1_2b,xTB,xTB,xTB,0.015272,5.169409,NaN,NaN
2,RS1_2b_to_PS1_3,xTB,xTB,xTB,56.181636,47.983556,NaN,NaN
3,RS2_1_to_TS2_1a,xTB,NaN,xTB,NaN,8.562170,NaN,NaN
4,TS2_1a_to_PS2_2,xTB,xTB,xTB,-16.087969,-72.221517,NaN,NaN


## Run one xTB test only after `xtb found:` is not `None`

## Optional constrained TS xTB test

## Run all xTB jobs only after the one-step test works

In [34]:
# Uncomment only when ready.
# run_xtb_jobs(list(STRUCTURE_SETS), roles=["RS", "TS", "PS"])

## Optional fast VeloxChem SMILES guesses

No relaxed QM scan. No 3D viewer.

In [35]:
# Uncomment only if VeloxChem is importable in this environment.
# run_veloxchem_fast(["RS2_1_to_TS2_1a"])

## Build the EVB-ready QM reference folder

Run this **after your QM outputs/charges are available**. It does not fabricate missing values; missing energies or charges are reported as missing.


In [36]:
# Build all folders and collect all available RS/TS/PS QM information
collect_qm_reference(list(STRUCTURE_SETS))
write_evb_comparison_template()



QM reference dataset prepared for EVB
Root: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm\qm_reference_for_evb
State summary: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm\qm_reference_for_evb\summaries\qm_state_summary.csv
Barrier table: D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_evb_qm\qm_reference_for_evb\summaries\qm_barriers_for_evb_comparison.csv
RS1_1_to_TS1_2               RS   ENERGY      CHARGES
RS1_1_to_TS1_2               TS   ENERGY      CHARGES
RS1_1_to_TS1_2               PS   ENERGY      CHARGES
TS1_2_to_PS1_2b              RS   ENERGY      CHARGES
TS1_2_to_PS1_2b              TS   ENERGY      CHARGES
TS1_2_to_PS1_2b              PS   ENERGY      CHARGES
RS1_2b_to_PS1_3              RS   ENERGY      CHARGES
RS1_2b_to_PS1_3              TS   ENERGY      CHARGES
RS1_2b_to_PS1_3              PS   ENERGY      CHARGES
RS2_1_to_TS2_1a              RS   ENERGY      CHARGES
RS2_1_to_TS2_1a              TS   ENERGY      CHARGES
RS2_1_to_TS2_1a              PS   ENERGY      CH